# Chapter 7: Unbiasedness, Consistency and the MSE Decomposition

The computational applications of this chapter of *Theory of Statistics*. The code is generated automatically from the book's `code/ch07/` files; see the book for the explanations and proofs.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibrahimguney/statistical-theory/blob/main/notebooks/ch_07.ipynb)

$$
\newcommand{\R}{\mathbb{R}}\newcommand{\N}{\mathbb{N}}\newcommand{\E}{\mathbb{E}}
\newcommand{\P}{\mathbb{P}}\newcommand{\F}{\mathcal{F}}\newcommand{\I}{\mathbb{I}}
\newcommand{\Var}{\operatorname{Var}}\newcommand{\Cov}{\operatorname{Cov}}
\newcommand{\Corr}{\operatorname{Corr}}\newcommand{\MSE}{\operatorname{MSE}}
\newcommand{\Bias}{\operatorname{Bias}}\newcommand{\Normal}{\mathcal{N}}
\newcommand{\iid}{\overset{\text{i.i.d.}}{\sim}}\newcommand{\pto}{\xrightarrow{P}}
\newcommand{\dto}{\xrightarrow{d}}\newcommand{\X}{\mathbf{X}}\newcommand{\x}{\mathbf{x}}
$$

In [1]:
# Setup: numpy, scipy and matplotlib are preinstalled on Colab. Locally:
#   pip install -r code/requirements.txt
# Figure files (PDF) saved by the code are written to the _figures/ folder; the figures are
# also shown inside the notebook.
import os
os.makedirs("_figures", exist_ok=True)
os.chdir("_figures")
%matplotlib inline

## Comparing the MSE of $S^2$ and the MLE

*Source file:* `code/ch07/c07_1_s2_vs_mle_mse.py`

In [2]:
import numpy as np

rng = np.random.default_rng(42)
n, s2_true = 5, 4.0
samples = rng.normal(10.0, np.sqrt(s2_true), size=(200_000, n))
s2 = np.var(samples, axis=1, ddof=1)
mle = np.var(samples, axis=1, ddof=0)
print(f"S^2 MSE: {np.mean((s2 - s2_true)**2):.4f} | MLE MSE: {np.mean((mle - s2_true)**2):.4f}")

# Family of divisors: SS / d , d = n-1, n, n+1 -- bias, variance, MSE
SS = np.sum((samples - samples.mean(axis=1, keepdims=True))**2, axis=1)
for d in [n - 1, n, n + 1]:
    est = SS / d
    print(f"divisor={d} | bias={est.mean()-s2_true:+.4f} | var={est.var():.4f} | MSE={np.mean((est-s2_true)**2):.4f}"
          f" | theory MSE={((d-n+1)**2 + 2*(n-1)) * s2_true**2 / d**2:.4f}")

S^2 MSE: 8.0765 | MLE MSE: 5.8004
divisor=4 | bias=+0.0067 | var=8.0765 | MSE=8.0765 | theory MSE=8.0000
divisor=5 | bias=-0.7946 | var=5.1689 | MSE=5.8004 | theory MSE=5.7600
divisor=6 | bias=-1.3289 | var=3.5895 | MSE=5.3554 | theory MSE=5.3333


## Estimators for $\text{Uniform}(0,\theta)$

*Source file:* `code/ch07/c07_2_uniform_estimators.py`

In [3]:
import numpy as np

rng = np.random.default_rng(42)
theta, R = 10.0, 100_000
print(" n   | MSE(2*Xbar)  theory   | MSE(X_max)  theory   | MSE((n+1)/n X_max) theory")
for n in [5, 20, 100]:
    X = rng.uniform(0, theta, size=(R, n))
    t1, t2 = 2*X.mean(axis=1), X.max(axis=1)
    t3 = (n + 1) / n * t2
    m = lambda t: np.mean((t - theta)**2)
    print(f"{n:4d} | {m(t1):8.4f} {theta**2/(3*n):8.4f} | {m(t2):8.4f} {2*theta**2/((n+1)*(n+2)):8.4f}"
          f" | {m(t3):8.4f} {theta**2/(n*(n+2)):8.4f}")

 n   | MSE(2*Xbar)  theory   | MSE(X_max)  theory   | MSE((n+1)/n X_max) theory
   5 |   6.6706   6.6667 |   4.7873   4.7619 |   2.8688   2.8571
  20 |   1.6550   1.6667 |   0.4315   0.4329 |   0.2262   0.2273
 100 |   0.3319   0.3333 |   0.0194   0.0194 |   0.0098   0.0098


## Inadmissibility: the James–Stein Estimator

For a single observation $\X\sim\Normal_k(\bm\theta,I_k)$ under total squared loss $\|\hat{\bm\theta}-\bm\theta\|^2$, the natural estimator $\hat{\bm\theta}=\X$ has risk $k$. Stein  showed that for $k\ge3$, $\X$ is *inadmissible*. Below, for $k=10$, the simulated risk of the James–Stein estimator  is compared with three values: the exact risk of Theorem 7.12, the upper bound of Corollary 7.13, and the risk $k$ of $\X$. The exact risk requires $\E_{\bm\theta}\|\X\|^{-2}$. $\|\X\|^2$ is a noncentral chi-square variable and, by a well-known representation, a Poisson mixture of central chi-squares: given $J\sim\text{Poisson}(\|\bm\theta\|^2/2)$, $\|\X\|^2\sim\chi^2_{k+2J}$. With the computation $\E[1/\chi^2_m]=1/(m-2)$ from the proof of the theorem, $\E_{\bm\theta}\|\X\|^{-2}=\E\big[1/(k-2+2J)\big]$.

*Source file:* `code/ch07/c07_3_james_stein.py`

In [4]:
import numpy as np
from scipy import stats

def exact_risk(norm_theta, k):
    # Theorem: R = k - (k-2)^2 E[1/||X||^2]; ||X||^2 is noncentral chi-square,
    # E[1/||X||^2] = E[1/(k-2+2J)], J ~ Poisson(||theta||^2 / 2)
    j = np.arange(400)
    expected = np.sum(stats.poisson.pmf(j, norm_theta**2 / 2) / (k - 2 + 2 * j))
    return k - (k - 2)**2 * expected

rng = np.random.default_rng(42)
k, R = 10, 200_000
for norm_theta in [0.0, 1.0, 3.0, 10.0]:
    theta = np.full(k, norm_theta / np.sqrt(k))
    X = theta + rng.normal(size=(R, k))
    js = (1 - (k - 2) / np.sum(X**2, axis=1, keepdims=True)) * X
    r_mle = np.mean(np.sum((X - theta)**2, axis=1))
    r_js = np.mean(np.sum((js - theta)**2, axis=1))
    upper = k - (k - 2)**2 / (k + norm_theta**2)
    print(f"||theta||={norm_theta:5.1f} | Risk(X)={r_mle:6.3f} | Risk(JS)={r_js:6.3f}"
          f" | exact={exact_risk(norm_theta, k):6.3f} | upper bound={upper:6.3f}")

||theta||=  0.0 | Risk(X)= 9.997 | Risk(JS)= 1.998 | exact= 2.000 | upper bound= 3.600
||theta||=  1.0 | Risk(X)= 9.998 | Risk(JS)= 2.732 | exact= 2.738 | upper bound= 4.182
||theta||=  3.0 | Risk(X)= 9.995 | Risk(JS)= 5.979 | exact= 5.986 | upper bound= 6.632


||theta||= 10.0 | Risk(X)= 9.992 | Risk(JS)= 9.394 | exact= 9.397 | upper bound= 9.418
